In [0]:
!pip install growwapi

In [0]:
from growwapi import GrowwAPI
from pyspark.sql.functions import *

In [0]:
user_api_key = "key"
user_secret = "secret"


access_token = GrowwAPI.get_access_token(api_key = user_api_key, secret = user_secret) 
# Use access_token to initiate GrowwAPi
groww = GrowwAPI(access_token)
# client = GrowwRestClient(access_token="NEW_TOKEN")
# print(client.get_user_profile())

In [0]:
def LoadData(start,end,symbol,interval=1):
    try:
        start_time = start+" 09:00:00"
        end_time = end+" 15:30:00"
        output=f"/Volumes/workspace/default/projecttradevolume/1_min/{symbol}/{end.replace('-','/')}/"
        historical_data_response = groww.get_historical_candle_data(
            trading_symbol=symbol,
            exchange=groww.EXCHANGE_NSE,
            segment=groww.SEGMENT_CASH,
            start_time=start_time,
            end_time=end_time,
            interval_in_minutes=interval
        )

        # print(historical_data_response['candles'])
        df=(spark
            .createDataFrame(historical_data_response['candles'],['DateTime','open','high','low','close','volume'])
            .withColumn('DateTime',from_utc_timestamp(from_unixtime("DateTime").cast("timestamp"),"Asia/Kolkata"))
            )
        df.write.parquet(output)
        print('Data Successfully Written into ',output)
    except Exception as E:
        print("No Data Available for that Date",E)

In [0]:
# LoadData('2026-01-20','2026-01-20','INFY')